# 01 – Exploratory Data Analysis (EDA)

This notebook downloads NASDAQ-100 data, inspects its structure, and creates
exploratory visualisations to understand the data before modelling.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.config import TICKER, START_DATE, END_DATE
from src.data.collector import DataCollector
from src.visualization.plotter import Plotter

print(f'Ticker : {TICKER}')
print(f'Period : {START_DATE} → {END_DATE}')

In [ ]:
# ── 1. Download stock data ──────────────────────────────────────────────────
collector = DataCollector()
df = collector.download_stock_data(save=True)
print(f'Shape : {df.shape}')
df.head()

In [ ]:
# ── 2. Basic statistics ─────────────────────────────────────────────────────
df.describe()

In [ ]:
# ── 3. Missing values ───────────────────────────────────────────────────────
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# ── 4. Price history plot ───────────────────────────────────────────────────
plotter = Plotter()
plotter.plot_price_history(df, title=f'{TICKER} Closing Price', filename='price_history.png')
print('Figure saved to reports/figures/price_history.png')

In [ ]:
# ── 5. Returns distribution ─────────────────────────────────────────────────
returns = df['Close'].pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(returns.index, returns, linewidth=0.5, color='steelblue')
axes[0].set_title('Daily Returns')
axes[0].set_ylabel('Return')
axes[0].grid(alpha=0.3)

axes[1].hist(returns, bins=80, color='steelblue', edgecolor='white')
axes[1].set_title('Return Distribution')
axes[1].set_xlabel('Daily Return')
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig('../reports/figures/returns_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Skewness : {returns.skew():.4f}')
print(f'Kurtosis : {returns.kurtosis():.4f}')

In [ ]:
# ── 6. Rolling volatility (30-day) ──────────────────────────────────────────
vol = returns.rolling(30).std() * (252 ** 0.5)  # annualised

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vol.index, vol, color='tomato', linewidth=0.8)
ax.set_title('30-Day Rolling Annualised Volatility')
ax.set_ylabel('Volatility')
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.savefig('../reports/figures/rolling_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7. Volume analysis ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(df.index, df['Volume'], width=1, color='grey', alpha=0.6)
ax.set_title('Daily Trading Volume')
ax.set_ylabel('Volume')
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Summary

- The NASDAQ-100 shows a general upward trend from 2020 to 2025 with two notable
  drawdowns (COVID crash 2020, tech correction 2022).
- Daily returns are approximately normally distributed with slight negative skew
  and fat tails (excess kurtosis > 0).
- Volatility spikes correlate with market stress events.

Continue to **02_feature_engineering.ipynb** to compute technical indicators.